# Video AI Concepts

**Module:** 18 — Video Generation

Temporal consistency, camera motion, latent time axes, conditioning, and a practical consistency toolkit.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain temporal consistency and common breakages
- Distinguish camera motion vs object motion vs morphing artifacts
- Use conditioning (image, pose, depth, audio) to stabilize generations
- Apply a consistency toolkit checklist in QA


## Temporal Consistency

### Definition
Temporal consistency means identity, geometry, lighting, and style remain stable across frames except for **intended** motion.

### Why it matters
Viewers forgive mild blur faster than identity soup or texture boil.

### How it works
Achieve via joint spatiotemporal generation, overlapping extends, control signals (pose/depth), low denoise video2video, and post stabilization.

### Intuition
A character must be the same actor walking — not a slideshow of cousins.

### Pitfalls
- Raising creativity sliders until identity dissolves
- Extending without locking seed/latents at the boundary

### When to use
Any multi-second generation meant to feel like one shot.


### Concepts map

| Concept | Meaning |
|---------|---------|
| Temporal attention | Model relates tokens across frames |
| Motion representation | Optical flow / trajectory latents |
| Keyframe conditioning | Pin start (and end) frames |
| Latent time axis | Noise/denoise across t and time |
| Guidance | Text/image CFG analogs for video |
| Autoregressive roll-out | Predict next frame/latent chunk |

```mermaid
flowchart TB
  K[Keyframe / prompt] --> M[Spatiotemporal generator]
  CTRL[Pose / depth / audio] --> M
  M --> F[Frame latents]
  F --> D[Decode + optional stabilize]
```


In [ ]:
# Demo 1: intended motion vs illegal jumps
def classify_delta(dx, dy, max_step=20):
    mag = (dx*dx + dy*dy) ** 0.5
    if mag <= max_step:
        return "plausible_motion"
    return "illegal_jump"

track = [(0,0), (5,2), (10,4), (80,4)]
for i in range(1, len(track)):
    dx = track[i][0]-track[i-1][0]
    dy = track[i][1]-track[i-1][1]
    print(i, classify_delta(dx, dy))


In [ ]:
# Demo 2: flicker metric on grayscale series (toy)
def flicker_index(frames):
    # mean abs diff of global mean luminance
    means = [sum(f)/len(f) for f in frames]
    diffs = [abs(means[i+1]-means[i]) for i in range(len(means)-1)]
    return sum(diffs)/len(diffs)

stable = [[0.4]*10, [0.41]*10, [0.39]*10]
boil = [[0.4]*10, [0.7]*10, [0.2]*10]
print(flicker_index(stable), flicker_index(boil))


## Camera vs Object vs Morph

### Definition
Separate **camera motion** (egomotion), **rigid/articulated object motion**, and **non-physical morphing**. Products should expose camera language deliberately.

### Why it matters
Users say 'pan left' and get soup because the model redistributes pixels without a camera model.

### How it works
Use camera tokens, motion brushes, trajectory controls, or img2video from storyboard plates with prompts that state camera explicitly.

### Intuition
Dolly moves the world across the sensor; morph melts the world.

### Pitfalls
- Stacking conflicting camera instructions
- Confusing style change with camera move

### When to use
Cinematic prompts, product turns, architecture walkthroughs.


### Consistency toolkit

| Tool | Use |
|------|-----|
| Image keyframe lock | Img2video from approved still |
| Pose / depth video | Transfer motion, keep structure |
| Low-strength video2video | Grade/style without rebuild |
| Overlap extends | Crossfade latent boundary |
| Face/ID adapters | Cast consistency |
| Optical-flow smoothers | Post flicker reduction |
| Manual NLE cuts | Hide impossible long takes |

```
ASCII:
  [storyboard stills] -> img2video clips -> extend -> NLE -> color -> audio
```


In [ ]:
# Demo 3: camera prompt normalizer
CAMERA_ALIASES = {
    "pan left": "camera pan left at constant speed",
    "orbit": "camera orbits subject 180 degrees",
    "fpv": "first-person drone forward flight",
    "static": "locked tripod, no camera motion",
}

def normalize_camera(prompt: str) -> str:
    p = prompt.lower()
    for k, v in CAMERA_ALIASES.items():
        if k in p:
            return f"{prompt.rstrip('.')}. {v}."
    return prompt + ". static camera."

print(normalize_camera("cat on a windowsill, pan left"))
print(normalize_camera("cat on a windowsill"))


In [ ]:
# Demo 4: conditioning bundle schema
def conditioning_bundle(text, image=None, pose_video=None, strength=0.7):
    return {
        "text": text,
        "image_url": image,
        "pose_video_url": pose_video,
        "video2video_strength": strength,
        "locks": [k for k,v in {"image": image, "pose": pose_video}.items() if v],
    }

print(conditioning_bundle("runner on trail", image="s3://k.png", pose_video="s3://pose.mp4"))


### Try it yourself — Concepts

1. Add a morph detector: large appearance change with near-zero optical-flow magnitude.
2. Write 10 camera-normalized prompts for a product turntable.
3. Design QA thresholds for flicker_index and identity drift.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `optical flow` | Per-pixel motion field between frames |
| `keyframe` | Authoritative frame anchoring a shot |
| `video2video` | Conditioned transform of an input clip |
| `temporal attention` | Attention spanning the time axis |


### Workshop — Parameter journal — Video AI Concepts

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Video AI Concepts
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Video AI Concepts

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Video AI Concepts
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Video AI Concepts

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Video AI Concepts
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Video AI Concepts

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Video AI Concepts
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Video AI Concepts

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Video AI Concepts
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Video AI Concepts

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Video AI Concepts
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Video AI Concepts

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Video AI Concepts
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Video AI Concepts

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Video AI Concepts
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Consistency is a first-class objective, not a polish pass
- Name camera motion explicitly; measure flicker and drift
- Toolkit: keyframes, controls, low-strength restyle, NLE honesty
